%md
###### Limpeza e Preparação na camada Silver

Neste notebook iremos realizar uma análise exploratória da nossa tabela para identificar os pontos de correção, limpeza e tratamento que devemos realizar na camada silver para prepararmos esses dados para serem analisados na próxima camada.

In [0]:
#Analisando se existem IDs nulos ou IDs duplicados em nossa base, para entender se existe algum problema sistêmico no registro dessas ocorrências.

df = spark.table("mvp_bernardomoraes_catalog.bronze.ocorrencias_cenipa")

total_linhas = df.count()
nulos_id = df.filter(df["Codigo_da_Ocorrencia"].isNull()).count()
distintos_id = df.select("Codigo_da_Ocorrencia").distinct().count()

print(f"Total de linhas: {total_linhas}")
print(f"IDs nulos: {nulos_id}")
print(f"IDs distintos: {distintos_id}")

Total de linhas: 6114
IDs nulos: 0
IDs distintos: 6114


🔍Como resultado de nossa primeira verificação, vimos que não existem IDs nulos ou duplicados, ou seja, não há necessidade de tratamento em cima desse tipo de problema.

Iremos verificar a seguir o nome e tipagem das colunas do nosso dataset.

In [0]:
#Primeiro, vamos verificar todos os nomes de colunas que existem.
df = spark.table("mvp_bernardomoraes_catalog.bronze.ocorrencias_cenipa")

for c in df.columns:
    print(c)

Codigo_da_Ocorrencia
Classificacao_da_Ocorrencia
Data_e_Hora_da_Ocorrencia
Latitude_da_Ocorrencia
Longitude_da_Ocorrencia
Cidade_da_Ocorrencia
UF_da_Ocorrencia
Pais_da_Ocorrencia
Aerodromo_da_Ocorrencia
Investigacao_da_Aeronave_foi_Liberada
Status_da_Investigacao
Numero_do_Relatorio_de_Divulgacao
Relatorio_de_Divulgacao_foi_Publicado?
Dia_da_Divulgacao_da_Publicacao
Total_de_Recomendacoes
Total_de_Aeronaves_Envolvidas
Ocorrencia_na_Saida_da_Pista?
Tipo_de_Ocorrencia
Tipo_de_Categoria_da_Ocorrencia
Taxonomia_do_Tipo_de_Icao
Matricula_da_Aeronave
Categoria_do_Operador_de_Aeronave
Tipo_de_Aeronave
Fabricante_da_Aeronave
Modelo_de_Aeronave
Tipo_de_Aeronave_Icao
Aeronave_Motor_Tipo
Aeronave_Motor_Quantidade
Aeronave_PMD
Categoria_PMD_Aeronave
Quantidade_de_Assentos_na_Aeronave
Ano_de_Fabricacao_da_Aeronave
Pais_Fabricante_da_Aeronave
Pais_de_Registro_da_Aeronave
Registro_da_Categoria_da_Aeronave
Registro_Segmento_da_Aeronave
Voo_de_Origem_do_Acidente
Voo_Destino_do_Acidente
Fase_de_Operacao

In [0]:
#Nesta etapa iremos gerar algumas sugestões de nomes com algumas boas práticas (como retirar o "?" e deixando os nomes todos com letra minúsculas, além de simplificar alguns nomes para facilitar)
import re

def sugerir_nome(col):
    nome = col.rstrip("?")                          # remove ? no final
    nome = nome.lower()                               # tudo minúsculo
    partes = nome.split("_")
    conectores = {"da", "de", "do", "no", "na", "dos", "das"}
    partes_limpas = [p for p in partes if p not in conectores]
    return "_".join(partes_limpas)

df = spark.table("mvp_bernardomoraes_catalog.bronze.ocorrencias_cenipa")

de_para = [(c, sugerir_nome(c)) for c in df.columns]
display(spark.createDataFrame(de_para, ["coluna_atual", "sugestao"]))

coluna_atual,sugestao
Codigo_da_Ocorrencia,codigo_ocorrencia
Classificacao_da_Ocorrencia,classificacao_ocorrencia
Data_e_Hora_da_Ocorrencia,data_e_hora_ocorrencia
Latitude_da_Ocorrencia,latitude_ocorrencia
Longitude_da_Ocorrencia,longitude_ocorrencia
Cidade_da_Ocorrencia,cidade_ocorrencia
UF_da_Ocorrencia,uf_ocorrencia
Pais_da_Ocorrencia,pais_ocorrencia
Aerodromo_da_Ocorrencia,aerodromo_ocorrencia
Investigacao_da_Aeronave_foi_Liberada,investigacao_aeronave_foi_liberada


In [0]:
#Antes de aplicar as mudanças, vamos apenas verificar se ocorreu de duas colunas ficarem com o mesmo nome sem querer:
from collections import Counter

sugestoes = [s for _, s in de_para]
repetidas = [nome for nome, qtd in Counter(sugestoes).items() if qtd > 1]
print("Colisões encontradas:", repetidas)

Colisões encontradas: []


In [0]:
#Por fim, iremos aplicar a transformação no nome das colunas
for atual, novo in de_para:
    df = df.withColumnRenamed(atual, novo)

for c in df.columns:
    display(c)

'codigo_ocorrencia'

'classificacao_ocorrencia'

'data_e_hora_ocorrencia'

'latitude_ocorrencia'

'longitude_ocorrencia'

...

'dia_assinatura_recomendacao'

'dia_encaminhamento_recomendacao'

'conteudo_recomendacao'

'status_recomendacao'

'orgao_recomendacao'

🔍Agora iremos verificar se o tipo das colunas está correto, para isso, vamos analisar as colunas de datas e booleanas que costumam ser mais críticas nesse sentido:

In [0]:
df.select("data_e_hora_ocorrencia", "dia_divulgacao_publicacao", 
           "dia_assinatura_recomendacao", "dia_encaminhamento_recomendacao") \
  .distinct() \
  .show(10, truncate=False)

df.select("investigacao_aeronave_foi_liberada", "relatorio_divulgacao_foi_publicado", 
           "ocorrencia_saida_pista") \
  .distinct() \
  .show(truncate=False) 

+----------------------+-------------------------+---------------------------+-------------------------------+
|data_e_hora_ocorrencia|dia_divulgacao_publicacao|dia_assinatura_recomendacao|dia_encaminhamento_recomendacao|
+----------------------+-------------------------+---------------------------+-------------------------------+
|07/02/2010 17:40      |2011-07-21               |2011-07-21                 |01/08/2011 00:00               |
|10/01/2010 23:15      |2011-06-30               |2011-06-30                 |01/08/2011 00:00               |
|06/02/2010 17:20      |2011-08-29               |2011-08-29                 |29/09/2011 00:00               |
|10/02/2010 09:03      |2016-04-22               |2016-04-22                 |17/05/2016 00:00               |
|04/02/2010 17:55      |2011-05-18               |2011-05-18                 |27/05/2011 00:00               |
|25/01/2010 16:20      |2012-11-19               |2012-11-19                 |11/12/2012 00:00               |
|

⚠️Vimos um ponto importante:Temos dados em nossas colunas booleanas além de "sim" e "não" (como nulo ou ***), vamos tratar essa informações transformando elas em nulas.

In [0]:
#Padronizando valores em colunas booleanas
def para_boolean(coluna):
    return when(col(coluna) == "SIM", True).when(col(coluna) == "NÃO", False).otherwise(None)

🔍Agora iremos analisar algumas das principais categorias para entender se precisamos realizar algum tratamento.

Primeiro, vamos começar com a mais crítica, a de fabricante:

In [0]:
#Analisando em um primeiro momento a coluna de fabricantes, que parecia ser a mais crítica em relação a preenchimento
from pyspark.sql.functions import count as spark_count

df.groupBy("fabricante_aeronave") \
  .agg(spark_count("*").alias("qtd")) \
  .orderBy("fabricante_aeronave") \
  .show(100, truncate=False)

+---------------------------------------+---+
|fabricante_aeronave                    |qtd|
+---------------------------------------+---+
|***                                    |355|
|ACS-INDUSTRIA AERONÁUTICA LTDA         |1  |
|AERO BOERO                             |201|
|AERO COMMANDER                         |2  |
|AEROALCOOL TECNOLOGIA LTDA             |3  |
|AEROBRAVO                              |1  |
|AEROBRAVO INDÚSTRIA AERONÁUTICA LTDA   |1  |
|AEROBRAVO LTDA                         |14 |
|AEROCENTRO AERONAVES                   |1  |
|AEROMOT                                |15 |
|AEROSPATIALE AND ALENIA                |291|
|AEROTEC                                |1  |
|AGUSTA                                 |59 |
|AHD                                    |1  |
|AILTON MARTINS DE OLIVEIRA             |1  |
|AIR TRACTOR                            |77 |
|AIR TRACTOR INC.                       |1  |
|AIRBUS                                 |3  |
|AIRBUS HELICOPTERS               

⚠️Como esperado, essa categoria é uma das mais problemáticas que precisamos tratar:
1- Existem alguns nomes de pessoas físicas (CARLOS ALBERTO VALENTE, FABIANO COSTA etc.), isso não é erro, é provavelmente uma aeronave amadora/experimental registrada no nome do construtor. Não conseguimos padronizar isso.

2- Temos algumas variações de grafias para o mesmo fabricantes em diversos casos, para isso, iremos corrigir utilizando um de-para.

3- Vimos que podem ter alguns fabricantes iguais mas registrados com ou sem acento, vamos tratar isso também.

In [0]:
#Começando por algumas boas práticas, vamos retirar os acentos dos registros
import unicodedata
from pyspark.sql.functions import udf, col

def remover_acentos(texto):
    if texto is None:
        return None
    return unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')

remover_acentos_udf = udf(remover_acentos)

df = df.withColumn("fabricante_aeronave", remover_acentos_udf(col("fabricante_aeronave")))

In [0]:
#Agora iremos deixar todos os registros com nomes maiúsculos, garantindo a padronização
from pyspark.sql.functions import upper, col

df = df.withColumn("fabricante_aeronave", upper(col("fabricante_aeronave")))

In [0]:
#Verificando se os tratamentos funcionaram
df.filter(col("fabricante_aeronave").like("%FABRICACAO PROPRIA%")).select("fabricante_aeronave").distinct().show(truncate=False)

+-------------------+
|fabricante_aeronave|
+-------------------+
|FABRICACAO PROPRIA |
+-------------------+



In [0]:
#Feitas as normalizações iniciais, vamos aplicar o de-para em casos que achamos em nossa exploração de fabricantes com formas de registros distintas
mapa_fabricante = {
    "AEROBRAVO LTDA": "AEROBRAVO",
    "AEROBRAVO INDUSTRIA AERONAUTICA LTDA": "AEROBRAVO",

    "AIRBUS INDUSTRIE": "AIRBUS",
    "EADS AIRBUS": "AIRBUS",

    "AIR TRACTOR INC.": "AIR TRACTOR",

    "BOEING COMPANY": "BOEING",

    "EDRA AERONAUTICA LTDA": "EDRA",
    "EDRA INDUSTRIA AERONAUTICA LTDA": "EDRA",

    "FLYER IND. AERONAUTICA LTDA": "FLYER",
    "FLYER INDUSTRIA AERONAUTICA LTDA": "FLYER",
    "FLYER INDUSTRIA AERONAUTICA": "FLYER",

    "FABRICANTE DESCONHECIDO": None,
}

from itertools import chain
from pyspark.sql.functions import create_map, lit, when, col

mapa_expr = create_map([lit(x) for x in chain(*mapa_fabricante.items())])

df = df.withColumn(
    "fabricante_aeronave",
    when(mapa_expr[col("fabricante_aeronave")].isNotNull(), mapa_expr[col("fabricante_aeronave")])
    .otherwise(col("fabricante_aeronave"))
)

In [0]:
#Verifcando o resultado
df.groupBy("fabricante_aeronave").agg(spark_count("*").alias("qtd")).orderBy("fabricante_aeronave").show(200, truncate=False)

+-----------------------------------------------+---+
|fabricante_aeronave                            |qtd|
+-----------------------------------------------+---+
|***                                            |355|
|ACS-INDUSTRIA AERONAUTICA LTDA                 |1  |
|AERO BOERO                                     |201|
|AERO COMMANDER                                 |2  |
|AEROALCOOL TECNOLOGIA LTDA                     |3  |
|AEROBRAVO                                      |16 |
|AEROCENTRO AERONAVES                           |1  |
|AEROMOT                                        |15 |
|AEROSPATIALE AND ALENIA                        |291|
|AEROTEC                                        |1  |
|AGUSTA                                         |59 |
|AHD                                            |1  |
|AILTON MARTINS DE OLIVEIRA                     |1  |
|AIR TRACTOR                                    |78 |
|AIRBUS                                         |323|
|AIRBUS HELICOPTERS         

⚠️ Vimos agora que o de-para funcionou, mas ainda temos valores "***" em nossos registros, como não são informações relevantes, vamos transformar em registros nulos, mas antes disso, vamos verificar se mais colunas necessitam desse tipo de tratamento.

In [0]:
#Verificando distribuição das outras colunas categóricas
colunas_categoricas = [
    "classificacao_ocorrencia",
    "tipo_ocorrencia",
    "fator_acidente",
    "area_fator",
    "aerodromo_ocorrencia",
    "uf_ocorrencia",
    "pais_ocorrencia",
    "tipo_aeronave",
]

for c in colunas_categoricas:
    print(f"=== {c} ===")
    df.groupBy(c).agg(spark_count("*").alias("qtd")).orderBy(c).show(50, truncate=False)

=== classificacao_ocorrencia ===
+------------------------+----+
|classificacao_ocorrencia|qtd |
+------------------------+----+
|ACIDENTE                |1930|
|INCIDENTE               |3393|
|INCIDENTE GRAVE         |791 |
+------------------------+----+

=== tipo_ocorrencia ===
+--------------------------------------------------------------------------------------------+---+
|tipo_ocorrencia                                                                             |qtd|
+--------------------------------------------------------------------------------------------+---+
|AERONAVE ATINGIDA POR OBJETO                                                                |12 |
|AERÓDROMO                                                                                   |6  |
|ALARME FALSO DE FOGO OU DE SUPERAQUECIMENTO                                                 |14 |
|CAUSADO POR FENÔMENO METEOROLÓGICO EM VOO                                                   |103|
|CAUSADO POR FENÔMENO MET

⚠️ Verificamos que existem alguns registros como "*" e outras variações que não indicam um preenchimento correto...vamos tratar esses casos deixando as linhas com esses dados como null.

In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace

for c, tipo in df.dtypes:
    if tipo == "string":
        df = df.withColumn(
            c,
            when(trim(col(c)).rlike(r"^[\*#!\?]+$"), None)
            .when(trim(col(c)) == "", None)                    
            .when(trim(col(c)) == "NULL", None)                 
            .otherwise(trim(col(c)))
        )

⚠️Por fim, na coluna "tipo_ocorrencia" verificamos a existência de um preenchimento incorreto, provavelmente por erro humano, onde temos "CONTATO ANORMAL COM A PISTAA", vamos corrigir esse caso.

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn("tipo_ocorrencia",
    when(col("tipo_ocorrencia") == "CONTATO ANORMAL COM A PISTAA", "CONTATO ANORMAL COM A PISTA")
    .otherwise(col("tipo_ocorrencia")))

✅ Agora que finalizamos os tratamentos, vamos escrever no schema silver a nossa tabela final tratada e, claro, mantendo a boa prática de descrevermos essa tabela.

In [0]:
df.write.mode("overwrite").saveAsTable("mvp_bernardomoraes_catalog.silver.ocorrencias_cenipa")

df_silver = spark.table("mvp_bernardomoraes_catalog.silver.ocorrencias_cenipa")
print("Linhas:", df_silver.count())
df_silver.printSchema()

Linhas: 6114
root
 |-- codigo_ocorrencia: integer (nullable = true)
 |-- classificacao_ocorrencia: string (nullable = true)
 |-- data_e_hora_ocorrencia: string (nullable = true)
 |-- latitude_ocorrencia: string (nullable = true)
 |-- longitude_ocorrencia: string (nullable = true)
 |-- cidade_ocorrencia: string (nullable = true)
 |-- uf_ocorrencia: string (nullable = true)
 |-- pais_ocorrencia: string (nullable = true)
 |-- aerodromo_ocorrencia: string (nullable = true)
 |-- investigacao_aeronave_foi_liberada: string (nullable = true)
 |-- status_investigacao: string (nullable = true)
 |-- numero_relatorio_divulgacao: string (nullable = true)
 |-- relatorio_divulgacao_foi_publicado: string (nullable = true)
 |-- dia_divulgacao_publicacao: date (nullable = true)
 |-- total_recomendacoes: integer (nullable = true)
 |-- total_aeronaves_envolvidas: integer (nullable = true)
 |-- ocorrencia_saida_pista: string (nullable = true)
 |-- tipo_ocorrencia: string (nullable = true)
 |-- tipo_categor

In [0]:
%sql
COMMENT ON TABLE mvp_bernardomoraes_catalog.silver.ocorrencias_cenipa 
IS 'Dados de ocorrências aeronáuticas do CENIPA tratados: colunas renomeadas (snake_case), placeholders (***, ####, NULL textual) convertidos em nulo real, datas e booleanos tipados, fabricante padronizado (sem acento, maiúsculo, de-para de variações de nome) e correções pontuais de digitação (ex: tipo de ocorrência). Fonte: bronze.ocorrencias_cenipa.';